<a href="https://colab.research.google.com/github/CharlieHubbard7/NFLResearch/blob/main/(6_2_26)_NFL_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NFL Research

This notebook contains my work for the third week of work with Professor Ron Yurko at Carnegie Mellon's Sports Analytics Center. I use Madden ratings to explore what stats teams value in players of different position groups

##Data Wrangling

### Installs and Imports


In [2]:
!pip install nflreadpy #This is from the NFLverse package: https://github.com/nflverse/nflreadpy
!pip install rapidfuzz -q #This helps match the NFL roster with the madden data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import string
import re
import math
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import nflreadpy as nfl
from rapidfuzz import fuzz, process
pd.set_option('display.max_columns', 250)
pd.set_option('display.min_rows', 10)

### NFL Verse Stats

In [13]:
pbp = nfl.load_pbp().to_pandas() # play-by-play data
player_stats = nfl.load_player_stats().to_pandas() # player game or season statistics
team_stats = nfl.load_team_stats().to_pandas() # team game or season statistics
# schedules = nfl.load_schedules().to_pandas() # game schedules and results
players = nfl.load_players().to_pandas() # player information
rosters = nfl.load_rosters().to_pandas() # team rosters
rosters_weekly = nfl.load_rosters_weekly().to_pandas() # team rosters by season-week
snap_counts = nfl.load_snap_counts().to_pandas() # snap counts
nextgen_stats = nfl.load_nextgen_stats().to_pandas() # advanced stats from nextgenstats.nfl.com
ftn_charting = nfl.load_ftn_charting().to_pandas() # charted stats from ftnfantasy.com/data
participation = nfl.load_participation().to_pandas() # participation data (historical)
draft_picks = nfl.load_draft_picks().to_pandas() # nfl draft picks
injuries = nfl.load_injuries().to_pandas() # injury statuses and practice participation
contracts = nfl.load_contracts().to_pandas() # historical contract data from OTC
officials = nfl.load_officials().to_pandas() # officials for each game
combine = nfl.load_combine().to_pandas() # nfl combine results
depth_charts = nfl.load_depth_charts().to_pandas() # depth charts
trades = nfl.load_trades().to_pandas() # trades
ff_playerids = nfl.load_ff_playerids().to_pandas() # ffverse/dynastyprocess player ids
ff_rankings = nfl.load_ff_rankings().to_pandas() # fantasypros rankings
ff_opportunity = nfl.load_ff_opportunity().to_pandas() # expected yards, touchdowns, and fantasy points

### Historical Madden Data

Credit to: https://github.com/theedgepredictor/nfl-madden-data/tree/main/data/madden/dataset for the data collection

In [4]:
madden_long = pd.read_parquet("/content/drive/MyDrive/ipynb stuff/madden_hist.parquet")
madden_long.head()

,player_id,madden_id,pfr_id,fullname,high_pos_group,position_group,position,season,team,last_season_av,overallrating,agility,acceleration,speed,stamina,strength,toughness,injury,awareness,jumping,trucking,archetype,runningstyle,changeofdirection,playrecognition,throwpower,throwaccuracyshort,throwaccuracymid,throwaccuracydeep,playaction,throwonrun,carrying,ballcarriervision,stiffarm,spinmove,jukemove,catching,shortrouterunning,midrouterunning,deeprouterunning,spectacularcatch,catchintraffic,release,runblocking,passblocking,impactblocking,mancoverage,zonecoverage,tackle,hitpower,press,pursuit,kickaccuracy,kickpower,return,year
0,00-0004543,SHANEDRONETT_d_line,DronSh20,Shane Dronett,def,d_line,DL,2001,ATL,1.0,75.0,53.0,65.0,58.0,79.0,84.0,84.0,79.0,81.0,43.0,7.347195,4,11,35.306958,66.518996,11.0,20.0,20.0,20.0,16.696763,19.318944,28.0,-2.261149,-1.546120,2.340851,3.478826,22.0,6.519022,10.549796,11.996476,9.005981,8.412261,8.034401,16.0,16.0,0.483380,15.249485,17.082539,81.0,51.752871,8.673651,66.456748,21.0,24.0,0.675884,2001
1,00-0012842,JASONPETER_d_line,PeteJa20,Jason Peter,def,d_line,DL,2001,CAR,1.0,74.0,66.0,74.0,64.0,84.0,78.0,84.0,80.0,67.0,51.0,7.889415,5,7,47.836493,58.864012,10.0,21.0,21.0,21.0,18.295792,21.480056,24.0,0.980120,-0.375920,8.238861,7.495244,21.0,5.982346,9.869347,12.073328,5.724684,4.637821,9.797546,27.0,18.0,9.782409,19.028072,23.221269,72.0,54.472249,12.652961,69.286862,18.0,19.0,3.772098,2001
2,00-0009978,LEONARDLITTLE_d_line,LittLe00,Leonard Little,def,d_line,DL,2001,LAR,1.0,73.0,72.0,62.0,75.0,65.0,70.0,70.0,82.0,64.0,52.0,25.090824,8,9,51.656422,55.302556,32.0,14.0,14.0,14.0,11.934877,13.800475,24.0,19.994085,18.021897,24.959818,21.045019,40.0,8.751804,11.499611,12.645619,17.544412,14.431167,17.109987,16.0,24.0,-5.096101,32.979157,37.209780,74.0,51.286358,28.416331,72.007990,11.0,33.0,1.131921,2001
3,00-0002142,JUNIORBRYANT_d_line,BryaJu20,Junior Bryant,def,d_line,DL,2001,SF,1.0,72.0,49.0,62.0,55.0,81.0,81.0,84.0,84.0,79.0,43.0,5.111116,5,9,36.576845,65.727568,24.0,11.0,11.0,11.0,9.494538,11.093699,14.0,-8.819708,-2.912042,-5.407254,-5.493794,14.0,7.008726,10.117281,12.240143,-1.108932,-1.319240,1.354661,29.0,27.0,7.181824,14.503342,17.728221,82.0,51.302609,6.509206,64.948565,12.0,14.0,1.162361,2001
4,00-0018054,REINARDWILSON_d_line,WilsRe20,Reinard Wilson,def,d_line,DL,2001,CIN,1.0,69.0,70.0,70.0,65.0,78.0,69.0,78.0,77.0,65.0,50.0,16.121217,7,9,50.852213,56.274011,31.0,16.0,16.0,16.0,16.041897,17.385281,26.0,12.748664,9.218156,18.243457,14.183115,26.0,7.190139,10.575620,11.980324,12.149089,9.289694,10.978948,13.0,9.0,-2.869764,26.808503,31.008670,69.0,51.400124,22.833778,69.883798,17.0,21.0,2.545349,2001


In [8]:

madden_long["rookie_year"] = madden_long.groupby("player_id")["year"].transform("min")
madden_long["years_in_nfl"] = madden_long["year"] - madden_long["rookie_year"] + 1
baseline = (madden_long[madden_long["years_in_nfl"] == 1]
            .set_index("player_id")["overallrating"])
madden_long["baseline_overall"] = madden_long["player_id"].map(baseline)
madden_long["overall_vs_baseline"] = madden_long["overallrating"] - madden_long["baseline_overall"]



## Analysis

### Exploring Variation of Players Rankings



In [18]:
temp = madden_long.query("rookie_year == 2020").sort_values("baseline_overall", ascending = False)
temp

,player_id,madden_id,pfr_id,fullname,high_pos_group,position_group,position,season,team,last_season_av,overallrating,agility,acceleration,speed,stamina,strength,toughness,injury,awareness,jumping,trucking,archetype,runningstyle,changeofdirection,playrecognition,throwpower,throwaccuracyshort,throwaccuracymid,throwaccuracydeep,playaction,throwonrun,carrying,ballcarriervision,stiffarm,spinmove,jukemove,catching,shortrouterunning,midrouterunning,deeprouterunning,spectacularcatch,catchintraffic,release,runblocking,passblocking,impactblocking,mancoverage,zonecoverage,tackle,hitpower,press,pursuit,kickaccuracy,kickpower,return,year,rookie_year,years_in_nfl,baseline_overall,overall_vs_baseline
50362,00-0036321,CHASEYOUNG_19990101,YounCh04,Chase Young,def,d_line,DL,2022,WAS,4.000000,86.000000,86.000000,91.000000,87.000000,80.000000,86.000000,89.000000,88.000000,86.000000,84.000000,34.000000,4,10,70.000000,83.000000,32.000000,28.000000,25.000000,20.000000,15.000000,33.000000,43.000000,36.000000,39.000000,36.000000,38.000000,33.000000,14.000000,6.611093,5.000000,24.000000,25.000000,17.000000,45.000000,45.000000,85.000000,34.000000,47.000000,89.000000,84.000000,27.000000,87.000000,22.000000,14.000000,10.000000,2022,2020.0,3.0,80.0,6.000000
53079,00-0036321,CHASEYOUNG_19990101,YounCh04,Chase Young,def,d_line,DL,2023,WAS,1.000000,85.000000,86.000000,91.000000,87.000000,80.000000,86.000000,89.000000,85.000000,84.000000,84.000000,34.000000,4,10,70.000000,81.000000,32.000000,28.000000,25.000000,20.000000,15.000000,33.000000,43.000000,36.000000,39.000000,36.000000,38.000000,33.000000,14.000000,8.799965,5.000000,24.000000,25.000000,17.000000,45.000000,45.000000,85.000000,34.000000,47.000000,89.000000,84.000000,27.000000,87.000000,22.000000,14.000000,10.000000,2023,2020.0,4.0,80.0,5.000000
44498,00-0036321,CHASEYOUNG_19990101,YounCh04,Chase Young,def,d_line,DL,2020,WAS,11.000000,80.000000,86.000000,91.000000,85.000000,80.000000,86.000000,89.000000,89.000000,74.000000,83.000000,34.000000,4,6,70.000000,71.000000,32.000000,28.000000,25.000000,20.000000,15.000000,33.000000,43.000000,36.000000,39.000000,36.000000,38.000000,33.000000,14.000000,9.943039,5.000000,24.000000,25.000000,17.000000,45.000000,45.000000,85.000000,34.000000,48.000000,83.000000,84.000000,27.000000,84.000000,22.000000,14.000000,10.000000,2020,2020.0,1.0,80.0,0.000000
56047,00-0036321,CHASEYOUNG_19990101,YounCh04,Chase Young,def,d_line,DL,2024,NO,2.000000,85.000000,86.000000,90.000000,86.000000,79.000000,86.000000,89.000000,83.000000,85.000000,84.000000,34.000000,4,10,71.000000,82.000000,32.000000,28.000000,25.000000,20.000000,15.000000,33.000000,43.000000,36.000000,39.000000,36.000000,38.000000,33.000000,14.000000,6.554689,5.000000,24.000000,25.000000,17.000000,45.000000,45.000000,85.000000,34.000000,47.000000,85.000000,84.000000,27.000000,86.000000,22.000000,14.000000,10.000000,2024,2020.0,5.0,80.0,5.000000
47530,00-0036321,CHASEYOUNG_19990101,YounCh04,Chase Young,def,d_line,DL,2021,WAS,14.000000,86.000000,86.000000,91.000000,85.000000,80.000000,86.000000,89.000000,89.000000,86.000000,84.000000,34.000000,4,10,70.000000,83.000000,32.000000,28.000000,25.000000,20.000000,15.000000,33.000000,43.000000,36.000000,39.000000,36.000000,38.000000,33.000000,14.000000,10.000000,5.000000,24.000000,25.000000,17.000000,45.000000,45.000000,85.000000,34.000000,48.000000,89.000000,84.000000,27.000000,87.000000,22.000000,14.000000,10.000000,2021,2020.0,2.0,80.0,6.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57687,00-0036120,ALEXTAYLOR_19970429,None,Armani Taylor-Prioleau,off,o_line,OL,2024,WAS,3.377049,60.466970,57.518819,68.593378,64.676871,80.657116,83.814191,81.738797,85.739980,64.850515,73.737413,29.060833,2,8,47.868787,22.149143,22.670554,9.144375,8.731648,8.499849,8.050666,8.003914,33.912983